In [2]:
from bs4 import BeautifulSoup
import os

In [4]:
os.chdir('../')

In [15]:
os.getcwd()

'c:\\Users\\Jovch\\Omnibus_Rebus\\Projects\\JobMarketIntelligence'

In [13]:
path = "apple_job_html/"
files = os.listdir(path)
print(len(files))

6047


In [51]:
'''
Extract text from HTML code
'''
def extract_text(html):
    soup = BeautifulSoup(html, 'html.parser')
    text_blocks = []
    for el in soup.find_all([
            "title",
            "h1", "h2", "h3", "h4", "h5", "h6",
            "p", "li", "blockquote"
    ]):
        text = el.get_text(" ", strip=True)
        
        if text:
            text_blocks.append({
                "tag": el.name,
                "text": text
            })
            
    return text_blocks


'''
Function to clean all the extra text information (footer, header, privacy etc)
'''
def clean_up(blocks):

    cleaned = []
    seen = set()
    for block in blocks:
        text = block["text"].strip().lower()

        if text in seen:
            continue

        seen.add(text)
        cleaned.append(text)
        
    return cleaned


In [80]:
import re

def find_job_title(soup):
    job_title = soup.find("h1", id="jobdetails-postingtitle")

    if job_title:
        return job_title.get_text(" ", strip=True)

    return None

def find_location(soup):

    location  = soup.find("label", id="jobdetails-joblocation")
    if location:
        return location.get_text(" ", strip=True)

def find_experience(text):
    years = []
    pattern = r'\b\d+(?:\s*-\s*\d+)?\+?\s*(?:years?|yrs?)\b'

    for sentence in text:
        matches = re.findall(pattern, sentence, re.IGNORECASE)

        for match in matches:
            numbers = re.findall(r'\d+', match)
            years.extend(int(num) for num in numbers)

    if years:
        return max(years)

    return "No Experience Nedded"

def find_education(text):
    education = []

    patterns = {
        r"\bbachelor(?:'s)?\b": "Bachelor's",
        r"\bb\.s\.?\b": "BS",
        r"\bb\.a\.?\b": "BA",
        r"\bmaster(?:'s)?\b": "Master's",
        r"\bm\.s\.?\b": "MS",
        r"\bm\.a\.?\b": "MA",
        r"\bassociate(?:'s)?\b": "Associate's",
        r"\ba\.s\.?\b": "AS",
        r"\ba\.a\.?\b": "AA",
        r"\bph\.?d\.?\b": "PhD",
        r"\bdoctorate\b": "Doctorate"
    }

    for sentence in text:
        for pattern, degree in patterns.items():
            if re.search(pattern, sentence, re.IGNORECASE):
                if degree not in education:
                    education.append(degree)
    return education

def find_industry(soup):
    industry = soup.find("label", id="jobdetails-teamname")

    if industry:
        return industry.get_text(" ", strip=True)

    return None

def parse_content(text, soup):
    content = {
        "job_title": find_job_title(soup),
        "location": find_location(soup),
        "experience": find_experience(text),
        "education": find_education(text),
        "industry": find_industry(soup)
    }

    print(content)

    return content

In [82]:
import json
from bs4 import BeautifulSoup

os.makedirs("apple_job_text", exist_ok=True)
text_dir = "apple_job_text"
content = []

for file in files:
    path = f"apple_job_html/{file}"
    with open(path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")
    text = extract_text(html)

    cleaned_text = clean_up(text)

    content.append(parse_content(cleaned_text, soup))

    with open(f"{text_dir}/{file}", "w") as f:
        json.dump(text, f, indent=4)

with open("data.json", "w") as f:
    json.dump(content, f, indent=4)
    

{'job_title': 'Wireless Systems Validation Engineer', 'location': 'Sunnyvale, California, United States', 'experience': 10, 'education': ["Master's", 'PhD'], 'industry': 'Hardware'}
{'job_title': 'PHY RTL Design Engineer', 'location': 'Sunnyvale, California, United States', 'experience': 'No Experience Nedded', 'education': [], 'industry': 'Hardware'}
{'job_title': 'Site Reliability Engineer, Enterprise Technology Services', 'location': 'Sunnyvale, California, United States', 'experience': 5, 'education': ["Bachelor's", "Master's"], 'industry': 'Software and Services'}
{'job_title': 'Enclosure Global Supply Manager', 'location': 'Shanghai, Shanghai, China', 'experience': 'No Experience Nedded', 'education': [], 'industry': 'Operations and Supply Chain'}
{'job_title': 'Early Career -  Power/EMIR Design Engineer (M, F, D)', 'location': 'Munich, Bavaria-Bayern, Germany', 'experience': 'No Experience Nedded', 'education': [], 'industry': 'Hardware'}
{'job_title': 'Software Systems Engineer

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
'''
TF-IDF weights
'''
def tf_idf(text):
    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(text)

    feature_names = vectorizer.get_feature_names_out()
    df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

    print(df_tfidf)